# Inference - Sentimen Analisis Roblox

### Import & Load

In [1]:
import numpy as np
import re
import pickle
import joblib
import nltk
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.tokenize import RegexpTokenizer

%pip install Sastrawi -q
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 1.7 MB/s eta 0:00:00


In [4]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Load model CNN+LSTM (model terbaik)
model_cnn_lstm = load_model('model_cnn_lstm.h5')

# Load tokenizer Keras
with open('tokenizer_dl.pkl', 'rb') as f:
    tokenizer_dl = pickle.load(f)

MAX_LEN = 100

print('Model dan tokenizer berhasil dimuat.')

Model dan tokenizer berhasil dimuat.


In [5]:
# Load kamus slang
kamus_slang = pd.read_excel('kamuskatabaku.xlsx')
slang_dict  = dict(zip(kamus_slang['tidak_baku'], kamus_slang['kata_baku']))

# Stemmer
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# Stopwords
stop_words   = set(stopwords.words('indonesian'))
regexp_token = RegexpTokenizer(r'\w+')

print('Kamus slang, stemmer, dan stopwords siap.')

Kamus slang, stemmer, dan stopwords siap.


### Fungsi Preprocessing

In [6]:
def preprocess_input(text):
    # Cleaning
    text = re.sub(r'https\S+', ' ', str(text), flags=re.IGNORECASE)
    text = text.lower()
    text = re.sub(r'@\S+', ' ', text)
    text = re.sub(r'#\S+', ' ', text)
    text = re.sub(r"'\w+", ' ', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    # Normalisasi slang
    text = ' '.join([slang_dict.get(w, w) for w in text.split()])

    # Stopword removal
    text = ' '.join([w for w in text.split() if w not in stop_words])

    # Tokenisasi
    tokens = regexp_token.tokenize(text)

    # Stemming
    tokens = [stemmer.stem(w) for w in tokens]

    # Filter pendek
    result = ' '.join([w for w in tokens if len(w) > 2])
    return result

### Fungsi Prediksi

In [7]:
label_map = {0: 'negatif', 1: 'netral', 2: 'positif'}

def prediksi_sentimen(teks):
    bersih = preprocess_input(teks)
    seq    = tokenizer_dl.texts_to_sequences([bersih])
    padded = pad_sequences(seq, maxlen=MAX_LEN, padding='post', truncating='post')
    proba  = model_cnn_lstm.predict(padded, verbose=0)[0]
    idx    = np.argmax(proba)
    return {
        'teks_asli'  : teks,
        'teks_bersih': bersih,
        'sentimen'   : label_map[idx],
        'probabilitas': {
            'negatif': f'{proba[0]:.4f}',
            'netral' : f'{proba[1]:.4f}',
            'positif': f'{proba[2]:.4f}'
        }
    }

### Testing Inferensi

In [8]:
contoh = [
    'game ini sangat seru dan menyenangkan banget, banyak mode permainan yang bisa dimainkan',
    'aplikasinya sering crash dan lemot parah, sangat mengecewakan',
    'lumayan sih tapi masih ada beberapa bug yang mengganggu'
]

for teks in contoh:
    hasil = prediksi_sentimen(teks)
    print(f'Teks     : {hasil["teks_asli"]}')
    print(f'Sentimen : {hasil["sentimen"].upper()}')
    print(f'Prob     : {hasil["probabilitas"]}')
    print('-' * 60)

Teks     : game ini sangat seru dan menyenangkan banget, banyak mode permainan yang bisa dimainkan
Sentimen : POSITIF
Prob     : {'negatif': '0.0410', 'netral': '0.0225', 'positif': '0.9365'}
------------------------------------------------------------
Teks     : aplikasinya sering crash dan lemot parah, sangat mengecewakan
Sentimen : NEGATIF
Prob     : {'negatif': '0.6432', 'netral': '0.1155', 'positif': '0.2414'}
------------------------------------------------------------
Teks     : lumayan sih tapi masih ada beberapa bug yang mengganggu
Sentimen : POSITIF
Prob     : {'negatif': '0.0628', 'netral': '0.0317', 'positif': '0.9055'}
------------------------------------------------------------


In [9]:
# Input manual
teks_input = input('Masukkan ulasan: ')
hasil = prediksi_sentimen(teks_input)

print(f'\nTeks     : {hasil["teks_asli"]}')
print(f'Sentimen : {hasil["sentimen"].upper()}')
print(f'Prob     : {hasil["probabilitas"]}')

Masukkan ulasan: gamenya mantap

Teks     : gamenya mantap
Sentimen : POSITIF
Prob     : {'negatif': '0.0421', 'netral': '0.0230', 'positif': '0.9349'}


### Batch Inferensi dari CSV

In [10]:
df_test = pd.DataFrame({
    'ulasan': [
        'roblox bagus banget gamenya seru',
        'sering error dan tidak bisa login sama sekali',
        'biasa aja sih tidak ada yang spesial',
        'update terbaru malah bikin lag parah',
        'senang banget bisa main bareng teman teman'
    ]
})

df_test['sentimen'] = df_test['ulasan'].apply(lambda x: prediksi_sentimen(x)['sentimen'])
df_test

,ulasan,sentimen
0,roblox bagus banget gamenya seru,positif
1,sering error dan tidak bisa login sama sekali,negatif
2,biasa aja sih tidak ada yang spesial,positif
3,update terbaru malah bikin lag parah,negatif
4,senang banget bisa main bareng teman teman,positif
